# choose 图片尺寸记录

扫描 `choose` 目录下的图片，记录宽度、高度（px）和长宽比，并保存为 CSV。

后续要加字段时，只改下面的 `extra_attributes`，返回的键会自动成为新列。

In [1]:
from pathlib import Path

import pandas as pd
from PIL import Image

ROOT = Path(r"D:\CouldService\Info Data")
CHOOSE_DIR = ROOT / "choose"
OUTPUT_CSV = ROOT / "choose_image_attributes.csv"

IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg", ".webp", ".bmp", ".gif", ".tif", ".tiff"}

## 属性

`aspect_ratio` = 宽度 / 高度。横图大于 1，竖图小于 1，正方形为 1。

在 `extra_attributes` 里追加字典即可，例如文件大小、色彩模式、主色。函数接收已打开的 `PIL.Image`，不用再读一次文件。

In [2]:
def size_attributes(image: Image.Image) -> dict:
    width, height = image.size
    return {
        "width_px": width,
        "height_px": height,
        "aspect_ratio": round(width / height, 6) if height else None,
    }


def extra_attributes(path: Path, image: Image.Image) -> dict:
    """后续新属性写在这里。返回的键会变成 CSV 列。"""
    return {
        # "file_size_bytes": path.stat().st_size,
        # "mode": image.mode,
    }


def record_image(path: Path) -> dict:
    relative = path.relative_to(CHOOSE_DIR)
    with Image.open(path) as image:
        row = {
            "relative_path": relative.as_posix(),
            "category": relative.parts[0] if len(relative.parts) > 1 else "",
            "filename": path.name,
        }
        row.update(size_attributes(image))
        row.update(extra_attributes(path, image))
    return row

In [3]:
paths = sorted(
    p for p in CHOOSE_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES
)

rows = [record_image(path) for path in paths]
df = pd.DataFrame(rows)

print(f"图片数量: {len(df)}")
df.head(10)

图片数量: 99


,relative_path,category,filename,width_px,height_px,aspect_ratio
0,Metaphor/1.png,Metaphor,1.png,2913,4409,0.660694
1,Metaphor/1075.png,Metaphor,1075.png,600,846,0.709220
2,Metaphor/1147.png,Metaphor,1147.png,1200,1200,1.000000
3,Metaphor/1153.png,Metaphor,1153.png,1130,816,1.384804
4,Metaphor/126.png,Metaphor,126.png,1920,2715,0.707182
5,Metaphor/1285.png,Metaphor,1285.png,1199,1499,0.799867
6,Metaphor/1422.png,Metaphor,1422.png,1427,2000,0.713500
7,Metaphor/1464.png,Metaphor,1464.png,1500,1491,1.006036
8,Metaphor/1601.png,Metaphor,1601.png,940,1861,0.505105
9,Metaphor/1604.png,Metaphor,1604.png,2568,948,2.708861


In [4]:
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(OUTPUT_CSV)

D:\CouldService\Info Data\choose_image_attributes.csv
